In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/exam-score-prediction-dataset/Exam_Score_Prediction.csv
/kaggle/input/playground-series-s6e1/sample_submission.csv
/kaggle/input/playground-series-s6e1/train.csv
/kaggle/input/playground-series-s6e1/test.csv


In [2]:
class CONFIG:
    INPUT_DIR = '/kaggle/input/playground-series-s6e1'

    SEED = 42
    FOLDS = 5
    TARGET = 'exam_score'

config = CONFIG()

In [3]:
train = pd.read_csv(f'{config.INPUT_DIR}/train.csv')
test = pd.read_csv(f'{config.INPUT_DIR}/test.csv')

train_org = pd.read_csv('/kaggle/input/exam-score-prediction-dataset/Exam_Score_Prediction.csv')
submission = pd.read_csv(f'{config.INPUT_DIR}/sample_submission.csv')

train['source'] = 'train'
test['source'] = 'test'
train_org['source'] = 'original'

cols = train.columns
cols = [col for col in cols if col not in ['id', 'student_id']]

test[config.TARGET] = np.nan

train = train[cols].copy()
test = test[cols].copy()
train_org = train_org[cols].copy()

combine = pd.concat([train_org, train, test], axis=0, ignore_index=True)


In [4]:
FEATURES = [col for col in cols if col not in [config.TARGET, 'source']]
CATS = combine[FEATURES].select_dtypes(include='object').columns.to_list()
CATS = [col for col in CATS if col not in ['source']]
NUMS = combine[FEATURES].select_dtypes(include=['int64', 'float64']).columns.to_list()

for df in [combine]:
    for col in NUMS:
        if df[col].dtype=='int64':
            df[col] = df[col].astype('int32')
        else:
            df[col] = df[col].astype('float32')

combine[CATS] = combine[CATS].astype('category')

In [5]:
combine[NUMS].nunique()

age                   8
study_hours         794
class_attendance    631
sleep_hours          70
dtype: int64

In [6]:
CATS1 = []

for c in NUMS:
    n = f'{c}_cat'
    for df in [combine]:
        df[n] = df[c].astype('category')
        print(df[n].dtype)

    CATS1.append(n)

print(CATS1)

category
category
category
category
['age_cat', 'study_hours_cat', 'class_attendance_cat', 'sleep_hours_cat']


In [7]:
combine.dtypes

age                        int32
gender                  category
course                  category
study_hours              float32
class_attendance         float32
internet_access         category
sleep_hours              float32
sleep_quality           category
study_method            category
facility_rating         category
exam_difficulty         category
exam_score               float64
source                    object
age_cat                 category
study_hours_cat         category
class_attendance_cat    category
sleep_hours_cat         category
dtype: object

In [8]:
CATS2 = []
SIZES = {}

for c in CATS+CATS1:
    n = f'{c}_enc'
    for df in [combine]:
        # df[c] = df[c].astype('category')
        df[n], _ = df[c].factorize()
        df[n] = df[n].astype('int32')
        s = df[n].max()+1

    CATS2.append(n)
    SIZES[n] = s

print(CATS2)
print('='*30)
print(SIZES)

['gender_enc', 'course_enc', 'internet_access_enc', 'sleep_quality_enc', 'study_method_enc', 'facility_rating_enc', 'exam_difficulty_enc', 'age_cat_enc', 'study_hours_cat_enc', 'class_attendance_cat_enc', 'sleep_hours_cat_enc']
{'gender_enc': 3, 'course_enc': 7, 'internet_access_enc': 2, 'sleep_quality_enc': 3, 'study_method_enc': 5, 'facility_rating_enc': 3, 'exam_difficulty_enc': 3, 'age_cat_enc': 8, 'study_hours_cat_enc': 794, 'class_attendance_cat_enc': 631, 'sleep_hours_cat_enc': 70}


In [9]:
train_idx = combine['source'] == 'train'
test_idx = combine['source'] == 'test'
train_org_idx = combine['source'] == 'original'

train_n = combine[train_idx].reset_index(drop=True).copy()
test_n = combine[test_idx].reset_index(drop=True).copy()
train_org_n = combine[train_org_idx].reset_index(drop=True).copy()

In [10]:
FEATURES_N = CATS2 + NUMS 
len(FEATURES_N)

X = train_n[FEATURES_N].copy()
y = train_n[config.TARGET].copy()

X_org = train_org_n[FEATURES_N].copy()
y_org = train_org_n[config.TARGET].copy()

test_n = test_n[FEATURES_N].copy()

def rmse(y, y_val):
    y_array = np.array(y)
    y_val_array = np.array(y_val)
    mse = np.mean((y_array - y_val_array)**2)
    rmse = np.sqrt(mse)
    return rmse

In [11]:
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
import gc
import warnings
warnings.filterwarnings('ignore')

params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'max_depth': 5,
    'colsample_bytree': 0.5,
    'subsample': 0.8,
    'n_estimators': 10000,
    'learning_rate': 0.01,
    'early_stopping_rounds': 100,
    'random_state': 42,
    'n_jobs': -1,
    'device': 'cuda',
    'enable_categorical': True,
}

kf = KFold(n_splits=config.FOLDS, shuffle=True, random_state=config.SEED)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test_n))


fold_scores = []
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_org_n = X_org.copy()
    y_org_n = y_org.copy()
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    # print(f'TRAINING SHAPE BEFORE DUPLICATION: {X_train.shape}')
    # for i in range(1):
    #     X_train = pd.concat([X_train, X_train], axis=0)
    #     y_train = pd.concat([y_train, y_train], axis=0)

    # for j in range(3):
    #     X_org_n = pd.concat([X_org_n, X_org_n], axis=0)
    #     y_org_n = pd.concat([y_org_n, y_org_n], axis=0)
    # print(f'TRAINING SHAPE AFTER DUPLICATION: {X_train.shape}')
    # X_train = pd.concat([train, X_org_n, X_train], axis=0, ignore_index=True)
    # y_train = pd.concat([train_y, y_org_n, y_train], axis=0, ignore_index=True)
    # print(f'TRAINING SHAPE :{X_train.shape}')
    # print(f'VALIDATION SHAPE :{X_val.shape}')
    
    # X_val = pd.concat([X_val])
    # print(f'TRAINING SHAPE BEFORE ENCODING :{X_train.shape}')
    # TE = TargetEncoder(cols_to_encode=CATS, aggs=['mean', 'count'], cv=5, smooth='auto', drop_original=False)
    # X_train = TE.fit_transform(X_train, y_train)
    # X_val = TE.transform(X_val)
    # test_enc = TE.transform(test_n)

    # TE2 = TargetEncoder(cols_to_encode=ROUND, aggs=['mean', 'count'], cv=5, smooth='auto', drop_original=True)
    # X_train = TE2.fit_transform(X_train, y_train)
    # X_val = TE2.transform(X_val)
    # test_enc = TE2.transform(test_n)
    # print(f'TRAINING SHAPE AFTER ENCODING :{X_train.shape}')
    model = XGBRegressor(**params)

    model.fit(X_train, y_train,
             eval_set=[(X_val, y_val)],
             verbose=1000)

    val_preds = model.predict(X_val)
    # oof_preds[val_idx] = val_preds

    fold_score = rmse(y_val, val_preds)
    fold_scores.append(fold_score)
    print(f'FOLD {fold} AUC: {fold_score:.4f}')
    test_preds +=  model.predict(test_n) / config.FOLDS


overall_rmse = np.mean(fold_scores)

print('='*30)
print(f"Overall OOF AUC: {overall_rmse:.4f}")
print('='*30)

[0]	validation_0-rmse:18.81910
[1000]	validation_0-rmse:8.79423
[2000]	validation_0-rmse:8.76025
[3000]	validation_0-rmse:8.73972
[4000]	validation_0-rmse:8.72670
[5000]	validation_0-rmse:8.71701
[6000]	validation_0-rmse:8.70985
[7000]	validation_0-rmse:8.70395
[8000]	validation_0-rmse:8.69965
[9000]	validation_0-rmse:8.69598
[9999]	validation_0-rmse:8.69314
FOLD 0 AUC: 8.6931
[0]	validation_0-rmse:18.86313
[1000]	validation_0-rmse:8.80739
[2000]	validation_0-rmse:8.77209
[3000]	validation_0-rmse:8.75002
[4000]	validation_0-rmse:8.73581
[5000]	validation_0-rmse:8.72575
[6000]	validation_0-rmse:8.71833
[7000]	validation_0-rmse:8.71200
[8000]	validation_0-rmse:8.70733
[9000]	validation_0-rmse:8.70378
[9999]	validation_0-rmse:8.70090
FOLD 1 AUC: 8.7009
[0]	validation_0-rmse:18.86870
[1000]	validation_0-rmse:8.79695
[2000]	validation_0-rmse:8.76015
[3000]	validation_0-rmse:8.73949
[4000]	validation_0-rmse:8.72520
[5000]	validation_0-rmse:8.71554
[6000]	validation_0-rmse:8.70839
[7000]	vali

In [12]:
submission[config.TARGET] = test_preds
submission.to_csv(f'submission_{overall_rmse}.csv', index=False)

In [13]:
imp_n = pd.DataFrame(
    {
        'feature_names': model.feature_names_in_,
        'gain': model.feature_importances_
    }
).sort_values('gain', ascending=False).reset_index(drop=True)

imp_n.to_csv('xgb_feature_importance.csv', index=False)
imp_n.head(10)

,feature_names,gain
0,study_hours,0.460530
1,sleep_quality_enc,0.172069
2,facility_rating_enc,0.117156
3,study_method_enc,0.097535
4,class_attendance,0.094371
5,sleep_hours,0.027839
6,study_hours_cat_enc,0.015434
7,class_attendance_cat_enc,0.002748
8,sleep_hours_cat_enc,0.002113
9,age_cat_enc,0.001827
